# 🧪 BioCirv AI Analysis Playground

Welcome to the BioCirv AI analysis environment. This notebook allows you to explore the biocirv project data using natural language queries powered by PandasAI and the CBORG LLM gateway.

## 🚀 Getting Started

### 1. Initialize Environment
Run the cell below to set up the connection to GCP and initialize the AI agent. This cell handles dependency installation, repository cloning, and authentication.

**⚠️ IMPORTANT: GOOGLE COLAB KERNEL**
This project requires **Python 3.11**. If the cell below reports Python 3.12+, please manually switch the runtime:
1. Go to **Runtime** -> **Change runtime type**.
2. Select **Python 3.11** from the Software version dropdown (if available).
3. Ensure the Hardware accelerator is set to **CPU** or **T4 GPU** (if needed).

**♻️ RESTART REQUIRED**: On the first run, this cell will uninstall Colab's default libraries and restart the session. **You must run this cell a second time** after the restart to complete the setup.

In [1]:
# 🚀 BioCirv AI - Nuclear Setup (Google Colab)

import os
import sys
import subprocess
import time
import socket

def is_colab():
    return 'google.colab' in sys.modules

# 1. Environment Verification
if is_colab():
    print(f"🔍 Python Version: {sys.version}")
    if sys.version_info >= (3, 12):
        print("\n❌ ERROR: Google Colab is running Python 3.12+.")
        print("PandasAI 3.0 dependencies (scipy 1.10.1) are currently incompatible with Python 3.12 binaries.")
        print("\n🛠️ FIX: Please switch to a Python 3.11 runtime:")
        print("   1. Click 'Runtime' in the top menu")
        print("   2. Select 'Change runtime type'")
        print("   3. Look for a 'Software version' or 'Python version' toggle")
        print("   4. Select 'Python 3.11'")
        print("   5. Click 'Save' and RE-RUN this cell.")
        sys.exit(1)

    # Check if we've already done the nuclear install in this session
    if not os.path.exists('/tmp/.biocirv_initialized'):
        print("☢️ Performing Nuclear Reset of scientific stack to ensure binary compatibility...")
        
        # Uninstall EVERYTHING that might conflict
        !pip uninstall -y pandasai pandas-ai pandas numpy scipy matplotlib pillow packaging -q
        
        # Force install the exact 'Legacy track' required by PandasAI 3.0
        !pip install --force-reinstall "pandas>=2.2.0" "numpy<2.0" "scipy>=1.10.1,<1.11" "matplotlib>=3.7.1,<3.8" "pillow>=10.1.0,<11.0.0" "packaging<25" -q
        
        # Mark as initialized
        with open('/tmp/.biocirv_initialized', 'w') as f: f.write('1')
        
        print("\n♻️ Restarting Runtime to load legacy binary stack... (The session will crash, this is normal)")
        print("👉 AFTER THE RESTART: Run this cell one more time to finish setup.")
        import os
        os.kill(os.getpid(), 9)

# --- If we reach here, we are in a clean Python 3.11 session with correct binaries ---

print("🔐 Authenticating with Google Cloud...")
try:
    from google.colab import auth
    auth.authenticate_user()
    print("✅ Authenticated successfully!")
except ImportError:
    print("ℹ️ Not in Colab environment, skipping auth.")

# Configuration - Using Port 5434 to avoid Colab default PG conflicts
os.environ['INSTANCE_CONNECTION_NAME'] = 'biocirv-470318:us-west1:biocirv-staging'
os.environ['DB_NAME'] = 'biocirv-staging'
os.environ['CLOUD_MODE'] = 'true'
os.environ['DB_HOST'] = '127.0.0.1'
os.environ['DB_PORT'] = '5434'
os.environ['DB_USER'] = 'biocirv_readonly'

print("🌐 Cloning repository...")
repo_path = '/content/biocirv-ai'
if os.path.exists(repo_path):
    !rm -rf {repo_path}
!git clone -b dev https://github.com/petercarbsmith/biocirv-ai.git -q

print("📦 Installing project and dependencies...")
!pip install --pre -e {repo_path} pg8000 cloud-sql-python-connector google-cloud-secret-manager -q

print("🌐 Setting up Cloud SQL Proxy...")
!rm -f cloud_sql_proxy
!curl -L -o cloud_sql_proxy https://storage.googleapis.com/cloud-sql-connectors/cloud-sql-proxy/v2.14.2/cloud-sql-proxy.linux.amd64 -q
!chmod +x cloud_sql_proxy

INSTANCE_CONNECTION_NAME = os.environ['INSTANCE_CONNECTION_NAME']

# Kill any existing proxy processes
!pkill -9 -f cloud_sql_proxy || true

print("🚀 Starting Cloud SQL Proxy (Hybrid Strategy: IAM Tunnel + Static Login)...")
proxy_process = subprocess.Popen(
    ['./cloud_sql_proxy', '--auto-iam-authn', '--port', '5434', INSTANCE_CONNECTION_NAME],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

def wait_for_proxy(host='127.0.0.1', port=5434, timeout=30):
    print(f"⏳ Waiting for Cloud SQL Proxy on {host}:{port}...")
    start_time = time.time()
    while time.time() - start_time < timeout:
        try:
            with socket.create_connection((host, port), timeout=1):
                print("✅ Cloud SQL Proxy is ready!")
                return True
        except (ConnectionRefusedError, socket.timeout):
            if proxy_process.poll() is not None:
                stdout, stderr = proxy_process.communicate()
                print(f"❌ Proxy process died!\nSTDERR: {stderr}")
                return False
            time.sleep(1)
    print("❌ Timeout waiting for proxy to start.")
    return False

proxy_ready = wait_for_proxy()

print("✅ Environment Synchronized!")

# 2. Configure Python Path
src_path = os.path.join(repo_path, "src")
if src_path not in sys.path:
    sys.path.append(src_path)

# 3. Secure Secret Retrieval
print("🔐 Retrieving secrets from Google Cloud Secret Manager...")
try:
    from google.cloud import secretmanager
    client = secretmanager.SecretManagerServiceClient()
    
    # Retrieve CBORG API Key
    if not os.getenv('CBORG_API_KEY'):
        try:
            name = "projects/biocirv-470318/secrets/CBORG_API_KEY/versions/latest"
            response = client.access_secret_version(request={"name": name})
            os.environ['CBORG_API_KEY'] = response.payload.data.decode("UTF-8")
            print("✅ CBORG API key retrieved securely.")
        except Exception as e:
            print(f"ℹ️ CBORG API Key retrieval skipped: {e}")
    
    # Retrieve Database Password (biocirv_readonly)
    try:
        pw_name = "projects/biocirv-470318/secrets/biocirv-staging-ro-biocirv_readonly/versions/latest"
        pw_response = client.access_secret_version(request={"name": pw_name})
        os.environ['DB_PASS'] = pw_response.payload.data.decode("UTF-8")
        print("✅ Database credentials retrieved securely.")
    except Exception as e:
        print(f"❌ CRITICAL: Could not retrieve database password: {e}")

except Exception as e:
    print(f"⚠️ Secret Manager error: {e}")

# 4. Initialize Sandbox & Agent
from ca_biositing.ai_exploration.sandbox_setup import init_sandbox, get_agent
llm, db_config = init_sandbox(cloud_mode=True)
agent = get_agent(llm, db_config)
print("\n✅ BioCirv AI Agent Ready!")

🔍 Python Version: 3.11.13 (main, Jun  4 2025, 08:57:29) [GCC 11.4.0]
🔐 Authenticating with Google Cloud...
✅ Authenticated successfully!
🌐 Cloning repository...
📦 Installing project and dependencies...
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 112.1 MB/s eta 0:00:0000:01:01
  Building editable for ca-biositing-ai-exploration (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
cudf-cu12 25.2.1 requires pandas<2

### 🔍 Connectivity Smoke Test
Verify raw database connectivity through the proxy tunnel before running AI queries.

In [2]:
# 🔍 Connectivity Smoke Test
import psycopg2
import os

print(f"Testing raw database connection to {os.environ.get('DB_HOST')}:{os.environ.get('DB_PORT')}...")
try:
    conn = psycopg2.connect(
        host=os.environ.get('DB_HOST', '127.0.0.1'),
        port=os.environ.get('DB_PORT', '5434'),
        user=os.environ.get('DB_USER', 'biocirv_readonly'),
        password=os.environ.get('DB_PASS'),
        dbname=os.environ.get('DB_NAME', 'biocirv-staging'),
        sslmode='disable' # Proxy handles encryption
    )
    print("✅ Success! Raw connection established via Proxy.")
    
    # Simple query check
    cur = conn.cursor()
    cur.execute("SELECT current_user, current_database();")
    user, db = cur.fetchone()
    print(f"📊 Connected as: {user} to database: {db}")
    
    cur.close()
    conn.close()
except Exception as e:
    print(f"❌ Connection Failed: {e}")
    print("\n💡 Troubleshooting Tips:")
    print("1. Check if the Cloud SQL Proxy is still running (pkill -0 cloud_sql_proxy).")
    print("2. Ensure you have authenticated with 'auth.authenticate_user()'.")
    print("3. Verify the secret 'biocirv-staging-ro-biocirv_readonly' exists in GCP Secret Manager.")

Testing raw database connection to 127.0.0.1:5434...
✅ Success! Raw connection established via Proxy.
📊 Connected as: biocirv_readonly to database: biocirv-staging


### 🛠️ Pre-flight Check
If you encounter issues, run this cell to verify the environment solve and module accessibility.

In [3]:
# 1. Verify Dependencies
print("🔍 Verifying AI Stack...")
import sys
print(f"🐍 Python: {sys.version}")
try:
    import pandas as pd
    import pandasai
    import pandasai_sql
    import pg8000
    from google.cloud.sql.connector import Connector
    print(f"✅ Pandas: {pd.__version__}")
    print(f"✅ PandasAI: {pandasai.__version__}")
    print(f"✅ SQL Connector: Found")
    print(f"✅ GCP Connector: Found")
except ImportError as e:
    print(f"❌ Missing Module: {e}")
    print("Please re-run the initialization cell above.")

# 2. Verify Repo Path
try:
    import ca_biositing
    print(f"✅ ca_biositing module loaded from: {ca_biositing.__file__}")
except ImportError:
    print("❌ ca_biositing NOT FOUND. Check your sys.path.")

🔍 Verifying AI Stack...
🐍 Python: 3.11.13 (main, Jun  4 2025, 08:57:29) [GCC 11.4.0]
✅ Pandas: 2.3.3
✅ PandasAI: <module 'pandasai.__version__' from '/usr/local/lib/python3.11/dist-packages/pandasai/__version__.py'>
✅ SQL Connector: Found
✅ GCP Connector: Found
✅ ca_biositing module loaded from: None


## 🔍 Starter Queries

Try running some of these queries to see the 'Trinity' output (Code, Data, Plot).

In [4]:
# Query 1: Data Summary
result = agent.chat("Show me a summary of the available views in the ca_biositing schema.")
result.display()

AttributeError: 'dict' object has no attribute 'model_dump'

In [ ]:
# Query 2: Visualization
result = agent.chat("Create a bar chart of the top 10 counties by biomass potential.")
result.display()

In [ ]:
# Query 3: Complex Analysis
result = agent.chat("Which counties have both high biomass potential and are within 50 miles of a major highway? Show the top 5.")
result.display()

## 🛠️ Advanced Usage

You can inspect the generated SQL and Python code for any query by looking at the `code` attribute of the result.